# ColonBench — Interactive Dataset Explorer

Browse colonoscopy videos and their annotations across all four ColonBench tasks:
**VQA (prompted & unprompted)**, **Classification**, and **Segmentation**.

Videos play inline; labels, questions, choices, and masks are printed alongside.

**Prerequisites** — make sure you have gated access, then either:
- run `hf auth login`, **or**
- set `HF_TOKEN` in your environment / `.env` file

In [ ]:
import os, io, base64, tempfile, subprocess
from pathlib import Path

from datasets import load_dataset
from huggingface_hub import hf_hub_download
from IPython.display import display, HTML, Image as IPImage
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import numpy as np

REPO_ID = "ajhamdi/colon-bench"

TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_API_KEY") or True

MASK_COLOR_BGR = (180, 50, 180)
MASK_ALPHA = 0.35


def download_video(video_id: str) -> str:
    """Download a video from the HF dataset and return the local cache path."""
    return hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset",
        filename=f"videos/{video_id}", token=TOKEN,
    )


def show_video(path: str, width: int = 640):
    """Embed an MP4 video in the notebook output."""
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video width="{width}" controls>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
        f'</video>'
    ))


def download_mask(mask_path: str) -> str:
    """Download a single mask PNG from the HF dataset."""
    return hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset",
        filename=mask_path, token=TOKEN,
    )


def apply_mask_overlay(
    frame: np.ndarray,
    mask: np.ndarray,
    color: tuple = MASK_COLOR_BGR,
    alpha: float = MASK_ALPHA,
) -> np.ndarray:
    """Blend a coloured semi-transparent mask onto a BGR frame."""
    if mask is None:
        return frame
    out = frame.copy()
    h, w = out.shape[:2]
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    overlay = np.zeros_like(out)
    overlay[:] = color
    mask_3ch = np.stack([mask, mask, mask], axis=2)
    return np.where(mask_3ch > 0, cv2.addWeighted(out, 1 - alpha, overlay, alpha, 0), out)


def _reencode_to_h264(src_path: str) -> str:
    """Re-encode mp4v to H.264 so browsers / notebook <video> tags can play it."""
    h264_path = src_path + ".h264.mp4"
    try:
        subprocess.run(
            [
                "ffmpeg", "-y",
                "-i", src_path,
                "-c:v", "libx264",
                "-preset", "fast",
                "-crf", "23",
                "-pix_fmt", "yuv420p",
                "-movflags", "+faststart",
                "-an",
                h264_path,
            ],
            check=True, capture_output=True, timeout=120,
        )
        os.replace(h264_path, src_path)
    except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired, OSError):
        if os.path.isfile(h264_path):
            try:
                os.remove(h264_path)
            except OSError:
                pass
    return src_path


def generate_overlay_video(
    video_path: str,
    mask_paths: list,
    mask_frame_indices: list,
) -> str:
    """Read a video, overlay masks on matching frames, write a browser-playable MP4."""
    mask_index = {}
    for mp, fi in zip(mask_paths, mask_frame_indices):
        local = download_mask(mp)
        raw = cv2.imread(local, cv2.IMREAD_GRAYSCALE)
        if raw is not None:
            mask_index[fi] = (raw > 127).astype(np.uint8)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 10.0

    out_path = os.path.join(tempfile.gettempdir(), f"overlay_{os.path.basename(video_path)}")
    writer = None
    frame_idx = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        mask = mask_index.get(frame_idx)
        if mask is not None:
            frame = apply_mask_overlay(frame, mask)
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
        writer.write(frame)
        frame_idx += 1

    cap.release()
    if writer:
        writer.release()

    return _reencode_to_h264(out_path)


print(f"Dataset: {REPO_ID}")
print("Helper functions ready: download_video, show_video, download_mask, generate_overlay_video")

In [ ]:
CONFIGS = ["vqa-prompted", "vqa-unprompted", "classification", "segmentation"]

datasets = {}
for cfg in CONFIGS:
    datasets[cfg] = load_dataset(REPO_ID, cfg, split="test")
    print(f"  {cfg:20s}  {len(datasets[cfg]):>5} samples   columns: {datasets[cfg].column_names}")

print(f"\nTotal: {sum(len(d) for d in datasets.values())} annotation rows across {len(CONFIGS)} configs")

## VQA — Visually Prompted

Each sample has a video with bounding-box overlays and a 5-way multiple-choice question.

In [ ]:
sample = datasets["vqa-prompted"][0]

print(f"Question ID : {sample['question_id']}")
print(f"Video       : {sample['video_id']}")
print(f"\nQ: {sample['question']}\n")
for letter in "ABCDE":
    marker = " ✓" if letter == sample["answer"] else ""
    print(f"  ({letter}) {sample[f'choice_{letter}']}{marker}")
print(f"\nCorrect answer: {sample['answer']}")

video_path = download_video(sample["video_id"])
show_video(video_path)

## VQA — Unprompted

Same question format, but the video has **no** bounding-box overlays — the model must localize the lesion on its own.

In [ ]:
sample = datasets["vqa-unprompted"][0]

print(f"Question ID : {sample['question_id']}")
print(f"Video       : {sample['video_id']}")
print(f"\nQ: {sample['question']}\n")
for letter in "ABCDE":
    marker = " ✓" if letter == sample["answer"] else ""
    print(f"  ({letter}) {sample[f'choice_{letter}']}{marker}")
print(f"\nCorrect answer: {sample['answer']}")

video_path = download_video(sample["video_id"])
show_video(video_path)

## Classification

Binary label: does the video contain a lesion (`1`) or not (`0`)?

In [ ]:
cls_ds = datasets["classification"]

label_names = {0: "no_lesion", 1: "lesion"}
pos = sum(1 for r in cls_ds if r["lesion"] == 1)
print(f"Classification split: {len(cls_ds)} videos  ({pos} lesion, {len(cls_ds)-pos} no-lesion)\n")

for idx in [0, len(cls_ds) // 2]:
    sample = cls_ds[idx]
    label = label_names[sample["lesion"]]
    print(f"--- Sample #{idx} ---")
    print(f"Video : {sample['video_id']}")
    print(f"Label : {sample['lesion']}  ({label})\n")
    video_path = download_video(sample["video_id"])
    show_video(video_path)

## Segmentation

Each sample has a video, a lesion description, and per-frame ground-truth masks.
Below we show the original video, a version with coloured mask overlays burned in, and a grid of sampled mask frames.

In [ ]:
sample = datasets["segmentation"][0]

print(f"Video       : {sample['video_id']}")
print(f"Description : {sample['description']}")
print(f"Mask frames : {sample['num_masks']}  (indices: {sample['mask_frame_indices'][:8]}{'...' if sample['num_masks'] > 8 else ''})")

video_path = download_video(sample["video_id"])

print("\n▶ Original video:")
show_video(video_path)

# Overlay video with masks burned in
print("\n▶ Video with mask overlay:")
overlay_path = generate_overlay_video(video_path, sample["mask_paths"], sample["mask_frame_indices"])
show_video(overlay_path)

# Show a grid of sampled masks
n_show = min(8, sample["num_masks"])
step = max(1, sample["num_masks"] // n_show)
indices = list(range(0, sample["num_masks"], step))[:n_show]

fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 2.5))
if n_show == 1:
    axes = [axes]
for ax, i in zip(axes, indices):
    mask_local = download_mask(sample["mask_paths"][i])
    mask_img = Image.open(mask_local)
    ax.imshow(mask_img, cmap="gray")
    ax.set_title(f"frame {sample['mask_frame_indices'][i]}", fontsize=9)
    ax.axis("off")
fig.suptitle(f"GT masks — {sample['description']}", fontsize=11)
plt.tight_layout()
plt.show()

## Browse any sample

Change `CONFIG` and `IDX` below to explore different samples.

In [ ]:
CONFIG = "vqa-prompted"   # "vqa-prompted" | "vqa-unprompted" | "classification" | "segmentation"
IDX = 5                   # row index to display

sample = datasets[CONFIG][IDX]
print(f"Config: {CONFIG}  |  Sample #{IDX}\n")

video_id = sample["video_id"]
print(f"Video: {video_id}")

if CONFIG.startswith("vqa"):
    print(f"\nQ: {sample['question']}\n")
    for letter in "ABCDE":
        marker = " ✓" if letter == sample["answer"] else ""
        print(f"  ({letter}) {sample[f'choice_{letter}']}{marker}")
    print(f"\nCorrect answer: {sample['answer']}")
elif CONFIG == "classification":
    label_names = {0: "no_lesion", 1: "lesion"}
    print(f"Label: {sample['lesion']}  ({label_names[sample['lesion']]})")
elif CONFIG == "segmentation":
    print(f"Description: {sample['description']}")
    print(f"Mask frames: {sample['num_masks']}")

video_path = download_video(video_id)

print("\n▶ Original video:")
show_video(video_path)

if CONFIG == "segmentation" and sample["num_masks"] > 0:
    print("\n▶ Video with mask overlay:")
    overlay_path = generate_overlay_video(video_path, sample["mask_paths"], sample["mask_frame_indices"])
    show_video(overlay_path)

    n_show = min(6, sample["num_masks"])
    step = max(1, sample["num_masks"] // n_show)
    indices = list(range(0, sample["num_masks"], step))[:n_show]
    fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 2.5))
    if n_show == 1:
        axes = [axes]
    for ax, i in zip(axes, indices):
        mask_local = download_mask(sample["mask_paths"][i])
        ax.imshow(Image.open(mask_local), cmap="gray")
        ax.set_title(f"frame {sample['mask_frame_indices'][i]}", fontsize=9)
        ax.axis("off")
    fig.suptitle(f"GT masks — {sample['description']}", fontsize=11)
    plt.tight_layout()
    plt.show()